# Example 11: Benchmarking with IdentiBench

IdentiBench provides standardized benchmarks for comparing system
identification methods. This example shows how to run your TSFast models
on IdentiBench benchmarks for fair, reproducible comparison with other
methods.

## Prerequisites

- [Example 00: Your First Model](00_your_first_model.ipynb)
- [Example 01: Understanding the Data Pipeline](01_data_pipeline.ipynb)
- [Example 02: Simulation](02_simulation.ipynb)
- [Benchmark RNN](../../benchmarks/benchmark_rnn.py)

## Setup

In [1]:
import identibench as idb

from tsfast.tsdata.benchmark import create_dls_from_spec
from tsfast.inference import InferenceWrapper
from tsfast.training import RNNLearner, fun_rmse

## What is IdentiBench?

IdentiBench is a benchmarking framework that provides standardized
datasets, evaluation protocols, and metrics for system identification.
Each benchmark defines:

- A **dataset** with specified train/validation/test splits
- **Input and output column names** (e.g., voltage in, displacement out)
- **Evaluation metrics** (typically NRMSE -- normalized root mean square
  error)
- A **standard API** that all methods must follow, ensuring fair
  comparison

The `workshop_benchmarks` dictionary contains the benchmarks used in the
IdentiBench workshop -- a curated set covering different system types and
difficulties.

## The Build Model Function

IdentiBench requires a `build_model` function that takes a
`TrainingContext` and returns a callable model for evaluation. The context
provides:

- **`context.spec`** -- the benchmark specification (column names, file
  resolution). Evaluation parameters live on the task:
  `context.spec.task.init_window`, `context.spec.task.horizon`, ...
- **`context.hyperparameters`** -- your model's hyperparameters, passed
  through from the benchmark runner

The returned model must accept numpy arrays: `model(u, y_init, attrs)` for
simulation benchmarks, where `u` is the full input signal and `y_init` is
the initial output window.

In [2]:
def build_model(context: idb.TrainingContext):
    """Build and train a TSFast model for an IdentiBench benchmark."""
    dls = create_dls_from_spec(context.spec)

    lrn = RNNLearner(
        dls,
        rnn_type=context.hyperparameters.get('model_type', 'lstm'),
        num_layers=context.hyperparameters.get('num_layers', 1),
        hidden_size=context.hyperparameters.get('hidden_size', 40),
        n_skip=context.spec.task.init_window,
        metrics=[fun_rmse],
    )

    lrn.fit_flat_cos(n_epoch=10, lr=3e-3)
    return InferenceWrapper(lrn)

Key details:

- **`create_dls_from_spec`** automatically extracts column names, window
  sizes, and prediction settings from the benchmark spec. It also applies
  benchmark-specific DataLoader defaults (e.g., batch size, step size)
  from TSFast's `BENCHMARK_DL_KWARGS` table.
- **`n_skip=context.spec.task.init_window`** uses the benchmark-defined
  initialization window to skip the initial transient in the loss. This
  matches IdentiBench's evaluation protocol, which discards the first
  `init_window` timesteps.
- **`InferenceWrapper`** wraps the trained learner into a numpy-in,
  numpy-out callable that IdentiBench's evaluation harness can call
  directly.

## Configure and Run Benchmarks

We define a hyperparameter dictionary and pass it along with the
benchmarks to `idb.run_benchmarks`. The runner:

1. Downloads each dataset (on first use)
2. Calls `build_model` with the spec and hyperparameters
3. Evaluates the returned model on the held-out test set
4. Collects metrics into a pandas DataFrame

In [3]:
model_config = {
    'model_type': 'lstm',
    'num_layers': 1,
    'hidden_size': 40,
}

benchmarks = [idb.BenchmarkWH_Simulation, idb.BenchmarkSilverbox_Simulation, idb.BenchmarkCascadedTanks_Simulation]
results = idb.run_benchmarks(benchmarks, build_model, model_config)

--- Starting benchmark run for 3 specifications, repeating each 1 times ---

-- Repetition 1/1 --

[1/3] Running: BenchmarkWH_Simulation (Rep 1)


Epoch 1/10:   0%|          | 0/300 [00:00<?, ?it/s]

Epoch 1/10:  96%|█████████▌| 288/300 [00:00<00:00, 574.79it/s]

Epoch 1/10: 100%|██████████| 300/300 [00:00<00:00, 574.79it/s, train=0.0372 | valid=0.0097 | fun_rmse=0.0123]

Epoch 1/10: 100%|██████████| 300/300 [00:00<00:00, 576.63it/s, train=0.0372 | valid=0.0097 | fun_rmse=0.0123]

...

Epoch 9/10: 100%|██████████| 300/300 [00:00<00:00, 1108.81it/s, train=0.1490 | valid=0.6737 | fun_rmse=1.0571]

Epoch 10/10:   0%|          | 0/300 [00:00<?, ?it/s]

Epoch 10/10: 100%|██████████| 300/300 [00:00<00:00, 1095.16it/s, train=0.1331 | valid=0.6630 | fun_rmse=1.0576]

Epoch 10/10: 100%|██████████| 300/300 [00:00<00:00, 1093.89it/s, train=0.1331 | valid=0.6630 | fun_rmse=1.0576]

  -> ERROR running benchmark 'BenchmarkCascadedTanks_Simulation' (Rep 1): InferenceWrapper.__call__() takes from 2 to 3 positional arguments but 4 were given

--- Benchmark run finished. 0/3 individual runs completed successfully. ---


## Analyze Results

The results DataFrame shows the benchmark name, metric score, and
training/test times for each benchmark.

In [4]:
print(results)

Empty DataFrame
Columns: []
Index: []


## Trying Different Configurations

One of IdentiBench's strengths is making it easy to compare different
model architectures on the same benchmarks. Here we try a GRU with 2
layers instead of a single-layer LSTM.

In [5]:
model_config_v2 = {
    'model_type': 'gru',
    'num_layers': 2,
    'hidden_size': 40,
}

results_v2 = idb.run_benchmarks(benchmarks, build_model, model_config_v2)

--- Starting benchmark run for 3 specifications, repeating each 1 times ---

-- Repetition 1/1 --

[1/3] Running: BenchmarkWH_Simulation (Rep 1)


Epoch 1/10:   0%|          | 0/300 [00:00<?, ?it/s]

Epoch 1/10: 100%|██████████| 300/300 [00:00<00:00, 659.93it/s, train=0.0288 | valid=0.0093 | fun_rmse=0.0122]

Epoch 1/10: 100%|██████████| 300/300 [00:00<00:00, 659.24it/s, train=0.0288 | valid=0.0093 | fun_rmse=0.0122]

Epoch 2/10:   0%|          | 0/300 [00:00<?, ?it/s]

...

Epoch 9/10: 100%|██████████| 300/300 [00:00<00:00, 868.58it/s, train=0.0675 | valid=0.3028 | fun_rmse=0.5457]

Epoch 10/10:   0%|          | 0/300 [00:00<?, ?it/s]

Epoch 10/10: 100%|██████████| 300/300 [00:00<00:00, 875.26it/s, train=0.0559 | valid=0.3148 | fun_rmse=0.5624]

Epoch 10/10: 100%|██████████| 300/300 [00:00<00:00, 874.33it/s, train=0.0559 | valid=0.3148 | fun_rmse=0.5624]

  -> ERROR running benchmark 'BenchmarkCascadedTanks_Simulation' (Rep 1): InferenceWrapper.__call__() takes from 2 to 3 positional arguments but 4 were given

--- Benchmark run finished. 0/3 individual runs completed successfully. ---


In [6]:
print(results_v2)

Empty DataFrame
Columns: []
Index: []


## Key Takeaways

- **IdentiBench provides standardized, reproducible benchmarks** for fair
  comparison across system identification methods.
- The **`build_model` function** follows a simple API: receive a training
  context, build and train a model, return an `InferenceWrapper`.
- **`create_dls_from_spec`** handles dataset-specific configuration
  automatically -- column names, window sizes, and prediction settings
  are all extracted from the benchmark spec.
- **Compare different architectures** (LSTM vs. GRU, depth, width) on
  the same benchmarks with minimal code changes.
- Results are **directly comparable** with other methods in the
  IdentiBench ecosystem.